# BiRefNet Fine-tune (Colab Pro A100) — Phase 3

This notebook runs the **Phase 3** (fine-tune) step of the `my-bg-remover`
project end-to-end on Google Colab (A100, 40GB): starting from the
`ZhengPeng7/BiRefNet_HR-matting` (MIT-licensed) initial weights, it fine-tunes
on the ~28k-pair BiRefNet-format dataset materialized under
`MyDrive/bg-remover-data/` in Phase 2 (`TRAIN/im/*.jpg` + `TRAIN/gt/*.png`,
plus `stats.json` + `train_composites_manifest.jsonl`).

**Why BiRefNet_HR-matting (and not RMBG-2.0)?** A licensing decision
(2026-07-09, `docs/superpowers/specs/2026-07-09-bg-remover-design.md`): the
finished model will be open-sourced under MIT. Fine-tuning from RMBG-2.0 would
inherit the CC BY-NC derivative-work risk. BiRefNet_HR-matting is MIT
licensed — chosen as the starting point both for license compatibility and for
its measured advantage on the camouflage category (see
the project's internal phase report (removed from the repo) §3: ~47% better MAE than rmbg-2.0).

## The RESEARCHED sources this notebook rests on (NO made-up config keys)

All of the files below were fetched with `curl` from the real
GitHub/HuggingFace content and **read** while writing this notebook
(2026-07-09, `ZhengPeng7/BiRefNet` `main` branch):

- `config.py` — `Config.__init__`: task selection (`task`), automatic
  `training_set` discovery (subfolders under `data_root_dir/{task}/*`,
  excluding `testsets`), `size=(1024,1024)` (default for the Matting task, and
  for all tasks except `General-2K`), `lambdas_pix_last` (per-task loss
  weights), `finetune_last_epochs` (Matting: **-10**), the `lr` formula
  (`(1e-4 if 'DIS5K' in task else 1e-5) * sqrt(batch_size/4)`),
  `mixed_precision=['no','fp16','bf16','fp8'][2]` (**bf16 default**), `compile=True`.
- `train.py` — `Trainer._train_batch` / `train_epoch`: loss computation order
  (pix loss + cls loss + gdt/out_ref loss), the `finetune_last_epochs`
  threshold (`epoch > args.epochs + config.finetune_last_epochs`), the
  checkpoint saving condition (`epoch >= args.epochs - config.save_last and
  epoch % config.save_step == 0` — only in the LAST few epochs, per the
  `val_last`/`step` values in `train.sh`).
- `dataset.py` — `MyData`: the `<data_root_dir>/<task>/<dataset>/{im,gt}`
  directory contract, `transform_image`/`transform_label` (ImageNet
  normalize); during training it returns the `(image, label, class_label)`
  triple.
- `models/birefnet.py` — `BiRefNet.forward`: returns a different output
  structure **depending on the `self.training` flag** (`[scaled_preds,
  class_preds_lst]` in training, only `scaled_preds` in eval — see the
  `model(inp)[-1].sigmoid()` pattern already used by `bgr/segmenter.py`, used
  IDENTICALLY in this notebook).
- `loss.py` — `PixLoss`/`ClsLoss`: which loss terms are ACTIVE (if
  `lambdas_pix_last[k]` is zero, that term is never computed).
- `utils.py` — `check_state_dict` (stripping the `module.`/`_orig_mod.`
  prefixes), `set_seed`.
- README (`Fine-tuning on Custom Data` section) — the official guide: put the
  data under `${data_root_dir}/{TASK}/{DATASET}/{im,gt}`, resume with
  `--resume` (the epoch number continues from the file name — `epoch_N.pth`);
  confirmed trainable directly on a single GPU (A100-40G) (in the author's own
  words).
- The HuggingFace model card (`ZhengPeng7/BiRefNet_HR-matting`) — **important
  finding:** this checkpoint was actually trained at **2048x2048**
  (`bb_pretrained: false` in `config.json`, card description "trained with
  images in 2048x2048"), while `BiRefNet-matting` (its non-HR sibling) is
  1024x1024. Loading method: `git clone` + `from models.birefnet import
  BiRefNet; BiRefNet.from_pretrained('ZhengPeng7/BiRefNet_HR-matting')`
  (code current from GitHub, weights from HuggingFace — the method the card
  itself recommends).

## DELIBERATE DEVIATIONS (why a custom loop INSTEAD of running the official `train.py` as-is?)

The real behavior of the official `train.py` + `train.sh` CONFLICTS with this
task's requirements in three places — so we import and use the official
modules (`config.py`, `dataset.py`, `models/birefnet.py`, `loss.py`,
`utils.py`) AS THEY ARE, but write our own thin loop instead of `train.py`'s:

1. **Checkpoint saving is too infrequent:** `train.py` saves a checkpoint only
   when `epoch >= args.epochs - config.save_last` (Matting: the last **50**
   epochs) AND `epoch % config.save_step == 0` (Matting: every **5** epochs) —
   in a 100-epoch run there is NO checkpoint at all between epochs 1-50. If
   the Colab session drops in that window, ALL progress is lost. The task
   requirement ("a checkpoint to Drive every epoch") demands otherwise — we
   save EVERY epoch.
2. **Gradient accumulation does NOT exist in the official code:** `train.py`
   is HARD-CODED with `Accelerator(..., gradient_accumulation_steps=1, ...)`;
   the `accelerator.accumulate(self.model)` context is left COMMENTED OUT in
   the file, unused. Since the A100-40G needs bs=2 physical + gradient
   accumulation (see the design note; BiRefNet issue #140: a known checkpoint
   bug at bs=1), we add it ourselves (the `ACCUM` parameter, manual
   `loss/ACCUM` scaling + periodic `optimizer.step()`).
3. **Checkpoints hold only model weights, NO optimizer/lr_scheduler state:**
   `train.py`'s `torch.save(...)` call saves only `state_dict()`. In a
   multi-session Colab training (dropping every few hours), resetting the
   momentum/lr schedule every time seriously hurts convergence — so we add
   `optimizer`/`lr_scheduler`/`epoch` to our checkpoint.

Apart from that, ALL other logic — the loss function order, the
`finetune_last_epochs` threshold, prefix cleanup with `check_state_dict` —
is kept IDENTICAL to the official code (line references above); only the
concrete conflict points in the three items were changed, and every one of
them is EXPLICITLY commented in these cells.

**This notebook was written WITHOUT being executed** (no Colab/GPU/Drive in
the development environment) — same method as
`training/prepare_data_colab.ipynb`: every cell is independent + annotated;
only static validation was done (JSON/nbformat + `ast.parse` of every cell,
plus a GPU-free simulation of the sampler/resume logic — see
`tests/test_train_colab_lib.py`). For surprises that may appear in the first
real Colab run (e.g. a `torch.compile` version incompatibility, the real
memory ceiling), the cells carry `try/except` + clear error messages.

## Parameters

Change the values below to your needs. **If you don't know ML**, read the
explanation next to each parameter — for most of them the default is already a
reasonable choice.

**v3 note:** RESUME is automatic — it continues from `epoch_2.pth` (v2) on
Drive, and with `EPOCHS=4` the 3rd and 4th epochs get trained. The rationale
for v3 rests on two real-data findings (see
the internal review notes (not in the repo)): (1) **domain gap** — the reason
over-deletion persisted on the real-photo benchmark was that all categories
EXCEPT camouflage were trained only on SYNTHETIC composited backgrounds;
`scripts/make_composites.py` now adds original-background copies (`_o00`) for
every category. (2) **the transparency target** — transparent MAE got WORSE
from v1 to v2 (0.0437->0.0481, ideogram target 0.0343);
`SAMPLER_PRESET="v3"` raises the transparent share from 20% to 24% (taken out
of the margin camouflage left in v2). Because the dataset grew by ~14k new
`_o00` pairs, the epoch length is pinned at PARITY with v2 (same Colab unit
cost) via `EPOCH_NUM_SAMPLES=27715`.

### COST TABLE — READ before raising `EPOCHS`

The cost of one epoch with the ~28k-pair dataset (A100, 1024px, bs=2 — a
theoretical estimate; once the FIRST epoch finishes, the MEASURED duration +
unit estimate is printed to the console):

| Item | Value |
|---|---|
| Iterations/epoch | ~13,700 (~27.4k train pairs / bs=2) |
| Time/epoch | **~1.7-3.4 hours** |
| Colab units/epoch | **~20-45 units** (A100 ~ 11-13 units/hour) |
| `EPOCHS=2` total | ~3.4-6.8 hours ~ **40-90 units** |
| `EPOCHS=100` total | ~170-340 hours ~ **2000-4000 units** — **NOT AFFORDABLE** on a monthly Colab Pro (~100 units) |

That is why the v2 default is `EPOCHS=2`: a short, affordable continuation
run — with `RESUME="auto"` it starts from `epoch_1.pth` (v1) on Drive, trains
2 more epochs with the new sampler balance (`SAMPLER_PRESET="v2"`), then
quality is measured with the 155-image LOCAL benchmark; if it works, buy more
units and continue from where it left off, again with `RESUME="auto"` (the
checkpoint/resume architecture was built exactly for this incremental
scenario). Decide on a longer run only knowing the cost.

Note: while `EPOCHS <= 10`, BiRefNet's "last 10 epochs loss-reweight"
fine-tune trick is skipped AUTOMATICALLY
(`train_colab_lib.should_apply_finetune_reweight` — in a short run the window
would overflow past epoch 1 and the decay exponent would be n>1 at the very
first epoch; details in the function docstring).


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # MEASURED OOM fix (v2) —
                                                                      # runs the CUDA memory allocator
                                                                      # with expandable segments to
                                                                      # reduce fragmentation; fixed the
                                                                      # OOM observed on A100-40GB at bs=2
                                                                      # in long runs (see the v2 prep
                                                                      # report). Must be set BEFORE torch
                                                                      # is imported — hence at the very
                                                                      # top of the file.

# --- Repo source: fill in EXACTLY ONE of the two (SAME rule as
#     prepare_data_colab.ipynb — a URL if you pushed the project to GitHub,
#     otherwise a zip you uploaded to Drive) ---
REPO_GIT_URL = ""  # e.g. "https://github.com/<user>/my-bg-remover.git"
REPO_ZIP_ON_DRIVE = ""  # e.g. "/content/drive/MyDrive/my-bg-remover.zip"

# --- Training parameters (summary for non-ML readers) ---
EPOCHS = 7            # v3: RESUME automatically continues from epoch_2.pth (v2) on Drive; this session
                      # trains the 3rd and 4th epochs. How many times the model sees the data end to end.
                      # CAUTION: every epoch costs ~1.7-3.4 hours / ~20-45 Colab units (read the COST
                      # TABLE above) — since EPOCH_NUM_SAMPLES is pinned (see below), the epoch cost
                      # stays at PARITY with v2 even if the dataset grows.
BATCH = 2             # v2: MEASURED, WORKING configuration. How many images are fed to the GPU at once
                      # per step. On A100-40GB, bs=2 is around the physical limit for 1024px + the
                      # Swin-L backbone (there is a known checkpoint bug at bs=1 — BiRefNet issue #140,
                      # see the design document — so do NOT use 1; see the NOTE below about WHY the
                      # `assert BATCH >= 2` was commented out).
ACCUM = 4             # v2: MEASURED, WORKING configuration. "Gradient accumulation": because GPU memory
                      # cannot hold a large group like bs=8 at once, we process ACCUM consecutive bs=2
                      # micro-groups, SUM their losses and update once — mathematically the effect is as
                      # if we trained with bs=(BATCH*ACCUM)=8 (the repo's own reference batch is 8, see
                      # the config.py `batch_size=8` comment).
LR = None             # Learning rate. None = the repo's official formula is computed automatically (see
                      # train_colab_lib.effective_lr — scaled by the effective batch), THEN multiplied by
                      # LR_SCALE (see below). If you don't know ML, DON'T TOUCH — leave None.
                      # NOTE: in sessions resumed via RESUME this parameter has NO EFFECT — the optimizer
                      # state in the checkpoint (including the old lr inside it) is restored; if you want
                      # to change the lr mid-run you have to update the optimizer param_groups by hand.
LR_SCALE = 0.5        # v2: MULTIPLIES the lr computed from the official formula
                      # (`train_colab_lib.effective_lr`) — applied only while LR=None (i.e. when the lr
                      # is auto-computed by the formula); if LR is set by hand (LR != None), LR_SCALE is
                      # NOT applied and the hand-entered value is used as-is. Default 0.5: we continue
                      # with a lower lr to reduce v1's forgetting — shrinking the lr in
                      # continuation/fine-tune runs reduces forgetting (standard in the literature).
                      # 1.0 = the old (v1) behavior, the formula is never scaled.
RESUME = "auto"        # "auto" = AUTOMATICALLY finds the latest checkpoint on Drive (or, failing that,
                      # on the local disk) and continues from it — in v2 that is epoch_1.pth on Drive:
                      # RESUME is automatic, continues from epoch_1.pth, the gains are preserved. If
                      # there is no checkpoint at all, it starts from SCRATCH from BiRefNet_HR-matting.
                      # Even if the Colab session drops, just re-running this notebook is enough.
DATA_DIR = "bg-remover-data"          # Name of the data folder on Drive (output of the Phase 2 notebook).
N_EVAL_EVERY = 2      # Every how many epochs a quick quality measurement (MAE) on a small validation
                      # sample (default 24 images) is run and written to train_log.txt. In the
                      # EPOCHS=4 v3 continuation run this means 2 measurement points (ends of the 3rd
                      # and 4th epochs).
EPOCH_NUM_SAMPLES = 27715        # v3: the FIXED number of samples WeightedRandomSampler draws per epoch
                                  # (see train_colab_lib.resolve_sampler_num_samples) — PARITY with v2's
                                  # epoch size (~27.4k train pairs / bs=2, COST TABLE above). Even if the
                                  # dataset grows by ~14k with the `_o00` copies, the epoch cost (~48
                                  # units) stays fixed — if left None (v1/v2 behavior),
                                  # len(train_dataset) is used as before.

# --- Advanced / rarely changed parameters ---
SEED = 42                      # The same seed, CONSISTENT with Phase 2 (VAL split + reproduction).
VAL_FRACTION = 0.02             # The share (2%) SET ASIDE from the training set, never used in training.
QUICK_EVAL_N = 24                # The FIXED number of images from the VAL set used in EVERY quick evaluation.
SAMPLER_PRESET = "v7"            # "v4" (default) = train_colab_lib.SAMPLER_PRESET_V4 (camouflage 12%,
                                  # transparent 18%, hair 8%, complex 19%, thin 13%, general 4%, text 10%,
                                  # fx 8%, illustration 8%) — tuned on v3's REAL benchmark results: the
                                  # focus is on complex+thin and the NEW capabilities (logo/text
                                  # preservation=text, around-object VFX glow=fx, illustration=illustration
                                  # — the data is produced by v4_veri_guncelleme_hucresi.py). transparent
                                  # KEPT at 18% (0.0043 away from Ideogram's 0.0343 target, the chase
                                  # continues); camouflage DOWN to 12% (v3 0.0304 vs Ideogram 0.1179 — the
                                  # margin is enormous); hair 8% (0.0067 MAE, close to rmbg's 0.0045 — the
                                  # share can be reduced). "v3"/"v2"/"v1" can still be selected for the old
                                  # behaviors — see cell (e); resolved via tcl.SAMPLER_PRESETS[SAMPLER_PRESET].
KEEP_LAST_N_CHECKPOINTS = 3      # Both locally and on Drive, only the last N epochs' checkpoints are kept (disk quota;
                                  # 3 bundles ~ 8-9GB of Drive space — see the training loop cell's Drive quota note).
NUM_WORKERS = 6                  # The DataLoader's parallel data-reading worker count. The official formula
                                  # (min(num_workers, batch_size)) would allow only 2 workers at bs=2 —
                                  # data reading becomes the bottleneck in a 13.7k-iteration epoch; with
                                  # A100 VMs having plenty of CPU cores we DECOUPLE this from the batch
                                  # (deliberate deviation).
COMPILE_MODEL = False            # torch.compile can speed up ~40% (config.py comment) but compilation is LAZY:
                                  # the try/except here cannot catch errors until the FIRST BATCH. False on
                                  # the first run (safe baseline); if you set True,
                                  # torch._dynamo.config.suppress_errors=True is also enabled (graph pieces
                                  # that fail to compile silently fall back to eager) — see the model setup
                                  # cell's note.
HF_MODEL_ID = "ZhengPeng7/BiRefNet_HR-matting"   # Initial weights (MIT) — NOT RMBG-2.0 (license decision).
BIREFNET_GIT_URL = "https://github.com/ZhengPeng7/BiRefNet.git"

DRIVE_ROOT = "/content/drive/MyDrive"
DRIVE_CKPT_SUBDIR = "bg-remover-checkpoints"     # every epoch is copied here (task requirement).
DRIVE_STATUS_SUBDIR = "bg-remover-status"        # train_log.txt is written here (the controller watches it from outside).
WORKDIR = "/content/my-bg-remover"               # where our repo gets cloned (for train_colab_lib.py + benchmark.metrics).
BIREFNET_DIR = "/content/BiRefNet"               # where the official BiRefNet repo gets cloned.
LOCAL_DATA_ROOT = "/content/dis_data"            # the local disk the Drive data is COPIED to (for I/O speed — reading through
                                                  # the Drive mount is very slow, see design doc §3, the "Colab session drop" note).
BIREFNET_TASK = "Matting"       # One of config.py's 6 ready-made tasks (DIS5K/COD/HRSOD/General/General-2K/Matting).
                                 # "Matting" was chosen because: (1) our initial weights (HR-matting) were trained on this task,
                                 # (2) its loss set (bce+mae+ssim, NO iou) handles the soft-alpha (transparent/hair) + hard-edge
                                 # (camouflage/general) mix in a balanced way, (3) size=(1024,1024) is already this task's
                                 # default — consistent with the testset resolution p50 (854px, see stats.json).

assert REPO_GIT_URL or REPO_ZIP_ON_DRIVE, "one of REPO_GIT_URL or REPO_ZIP_ON_DRIVE must be filled in"
# assert BATCH >= 2, "there is a known checkpoint bug at bs=1 (BiRefNet issue #140) — use BATCH>=2"
# NOTE (v2): the assert above was REMOVED (commented out, not deleted). PYTORCH_CUDA_ALLOC_CONF
# (at the top of this cell) was added as a measured OOM fix, but on some VMs/sessions BATCH=2 can
# still be marginal; the hard constraint was removed so the user can consciously try BATCH=1
# (the checkpoint-bug risk of issue #140 STILL APPLIES — it is just a user choice now, no longer
# ENFORCED by the notebook). The default is still BATCH=2 (the measured, working configuration) —
# consider this line only if OOM persists and you knowingly want to try BATCH=1.
print("Parameters set.")


## (a) Mount Drive + Fetch the Repos

Two repos get cloned: **BiRefNet** (the official training code — `config.py`,
`train.py`, `dataset.py`, `models/`, `loss.py`, `utils.py`) and **our own
project** (`my-bg-remover` — only `training/train_colab_lib.py` — the pure
Python sampler/resume/checkpoint-pruning logic, not copied or rewritten so it
stays the single source of truth — and `benchmark/metrics.py` — for the
quick-evaluation MAE, the SAME function used across the project).

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

os.makedirs(f"{DRIVE_ROOT}/{DRIVE_CKPT_SUBDIR}", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/{DRIVE_STATUS_SUBDIR}", exist_ok=True)
print("Drive mounted.")

In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path


def _clone_or_extract(repo_git_url: str, repo_zip_on_drive: str, workdir: str) -> None:
    if Path(workdir).exists():
        print(f"{workdir} already exists — skipping clone/extract (idempotent).")
        return
    if repo_git_url:
        print(f"git clone {repo_git_url} -> {workdir}")
        subprocess.run(["git", "clone", repo_git_url, workdir], check=True)
    else:
        print(f"extracting zip: {repo_zip_on_drive} -> {workdir}")
        extract_tmp = f"{workdir}_zip_extract_tmp"
        with zipfile.ZipFile(repo_zip_on_drive) as zf:
            zf.extractall(extract_tmp)
        extracted = list(Path(extract_tmp).iterdir())
        src_root = extracted[0] if len(extracted) == 1 and extracted[0].is_dir() else Path(extract_tmp)
        shutil.move(str(src_root), workdir)


# Our own project: only needed for train_colab_lib.py (sampler/resume logic) + the
# benchmark/ package (MAE) — pip install -e . makes the "benchmark" package importable.
_clone_or_extract(REPO_GIT_URL, REPO_ZIP_ON_DRIVE, WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
os.chdir(WORKDIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=True)
import training.train_colab_lib as tcl  # noqa: E402  (the sys.path insertion above + namespace-package import)
import benchmark.metrics as bm  # noqa: E402

# The official BiRefNet repo.
_clone_or_extract(BIREFNET_GIT_URL, "", BIREFNET_DIR)

# CAUTION: BiRefNet's requirements.txt (GitHub main branch, verified) contains
# `torch>=2.5.0`, `torchvision` and `numpy<2` lines. Trying to reinstall Colab's
# pre-installed torch (a build MATCHED to the image's own CUDA version) and
# numpy 2.x accordingly is two separate traps: (1) for torch, the risk of a
# CPU-only wheel / CUDA version mismatch, (2) `numpy<2` DOWNGRADES Colab's
# numpy 2.x — installed packages previously compiled against numpy 2 break and
# you end up in a "restart runtime" loop. So we STRIP the torch/torchvision/numpy
# lines and install the rest — we do NOT touch Colab's own (already compatible)
# installations.
req_path = Path(BIREFNET_DIR) / "requirements.txt"
filtered = [
    line for line in req_path.read_text().splitlines()
    if line.strip() and not line.strip().lower().startswith(("torch", "numpy"))
]
filtered_req_path = Path("/content/birefnet_requirements_filtered.txt")
filtered_req_path.write_text("\n".join(filtered) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(filtered_req_path), "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "safetensors", "-q"], check=True)
print("Repos ready:", WORKDIR, BIREFNET_DIR)
print(f"torch version (Colab's own, UNCHANGED): {__import__('torch').__version__}")
print(f"numpy version (Colab's own, UNCHANGED): {__import__('numpy').__version__}")

## (b) Patch `config.py` (task + path + batch)

BiRefNet's official fine-tuning guide (`README.md` "Fine-tuning on Custom
Data") recommends changing a few variables inside `config.py` BY HAND
(`sys_home_dir`, task selection, `training_set`/`testsets`). Since we use the
**existing** `Matting` task, we do NOT INVENT a new task name (the guide
generally says "pick a new task name", but one of the 6 ready-made tasks
already fits our need exactly — see the `BIREFNET_TASK` note in the parameters
cell) — we only change three lines textually: (1) we switch the selected index
of the `self.task` list to `Matting`, (2) we point `sys_home_dir` at our local
disk (`LOCAL_DATA_ROOT`) — so `data_root_dir` points at the fast local disk
instead of Drive's SLOW mount point, (3) we set `batch_size` equal to the
`BATCH` parameter (which also triggers the `config.lr`/`config.num_workers`
computations correctly). `self.bb` (the backbone selection, `swin_v1_l`) is
NEVER touched — the HR-matting weights were trained with this backbone; if it
changed, `from_pretrained` would try to load a state_dict into the wrong
architecture.

The patch logic lives in `train_colab_lib.apply_config_patches` (regex-based,
**idempotent** and re-parameterizable — re-running this cell on the same VM,
with the same OR different parameter values, does not blow up; unit-tested,
see `tests/test_train_colab_lib.py::test_apply_config_patches_is_idempotent`).
If a pattern is not found at all on the `main` branch (the repo changed), it
raises a clear `ValueError` — it does not silently misbehave.

In [ ]:
import re

config_py_path = Path(BIREFNET_DIR) / "config.py"
local_home = str(Path(LOCAL_DATA_ROOT))  # data_root_dir will be LOCAL_DATA_ROOT/datasets/dis (SAME root as cell (c))
patched = tcl.apply_config_patches(
    config_py_path.read_text(), task=BIREFNET_TASK, sys_home_dir=local_home, batch_size=BATCH
)
config_py_path.write_text(patched)
print(f"config.py patched (idempotent): task={BIREFNET_TASK}, sys_home_dir={local_home}, batch_size={BATCH}")

## (c) Copy the Data to Local Disk + Deterministic TRAIN/VAL Split

Reading thousands of small files through the Drive mount is VERY SLOW (as the
design document also notes) — so we copy the data ONCE to the local disk
(`/content`, Colab's ephemeral but FAST disk). Per the `config.py` contract
the target path is: `<data_root_dir>/<TASK>/TRAIN/{im,gt}` (BiRefNet's
`MyData` class auto-discovers the subfolders named in `training_set` — see the
`config.py` `datasets_all` line).

**The VAL split is NOT a physical move; it is a deterministic + PERSISTENT
SELECTION + separate copy**: `train_colab_lib.load_or_create_val_split` writes
the val list chosen on the FIRST run to Drive
(`bg-remover-status/val_stems.json`); ALL subsequent runs read from that file.
Why persistence is needed: the dataset on Drive can GROW later (the Phase 2
pipeline is idempotent — a new run may add new pairs); a purely deterministic
split would produce a DIFFERENT val set once the input list changed, and
images seen in training during earlier epochs would leak into val.
**Documented choice:** ALL stems added later go to TRAIN — the val set stays
whatever was chosen on the first run (a cross-epoch comparable MAE trend is
worth more than growing val proportionally; the quick evaluation already uses
a fixed subset of 24).

The VAL pairs are **NOT COPIED into `Matting/TRAIN`** (they are excluded from
the training set entirely — no leak) and are placed in a separate
`val_holdout/{im,gt}` folder — DELIBERATELY OUTSIDE the
`<data_root_dir>/Matting/` tree (had we put it as a "sibling" folder inside
the same tree, `config.py`'s `datasets_all` auto-discovery would mistake it
for a training source and include it in `training_set` — see the next cell's
comment). `MyData` never sees this folder; only our periodic quick evaluation
uses it.

Idempotency: if the target file already exists AND the file sizes of **both
im and gt** exactly match the source, it is NOT re-copied
(`train_colab_lib.copy_pairs`, unit-tested) — gt files left truncated by an
interrupted copy are also repaired automatically on the next run.

**Fast path (tar shards):** if `bg-remover-data/tar/_manifest.json` exists on
Drive (produced ONCE by running `training/veri_tar_paketleme_hucresi.py` on a
free CPU Colab), the 52k+ small files are not copied one by one — a small
number of large tar shards are copied from Drive to local
(`shutil.copy2` + validation against the byte size in the manifest), extracted
into `TRAIN/` (`tarfile.extractall(filter="data")`), and the number of
extracted files is compared with the manifest's `total_pairs` value (mismatch
= RuntimeError). Then the stems in `val_stems.json` are MOVED from TRAIN to
`val_holdout` (the persistent-split behavior is preserved exactly) and the new
pairs added to Drive AFTER the tar packing (the delta) are completed with
`copy_pairs` — the rule is preserved: new stems always go to TRAIN. The tar
path is idempotent too: if the expected number of files already exists
locally, the download/extract is skipped (session restart safety). If there is
no manifest, the old file-by-file path works UNCHANGED (~75 min; ~10 min via
the tar path).

In [ ]:
import json
import shutil
import tarfile
import time

drive_data_dir = Path(DRIVE_ROOT) / DATA_DIR
drive_train_im = drive_data_dir / "TRAIN" / "im"
drive_train_gt = drive_data_dir / "TRAIN" / "gt"
assert drive_train_im.is_dir() and drive_train_gt.is_dir(), (
    f"Expected data not found on Drive: {drive_train_im} / {drive_train_gt} — "
    f"prepare_data_colab.ipynb (Phase 2) may not have completed yet."
)

def _iterdir_retry(d, attempts=5, wait_s=30):
    """Drive FUSE occasionally throws a transient 'Errno 5 I/O error' on
    directories with 50k+ files (seen in the v3/v4 runs) - waits and retries."""
    import time
    for i in range(attempts):
        try:
            return list(d.iterdir())
        except OSError as e:
            if i == attempts - 1:
                raise
            print(f"WARNING: {e} while listing {d} - will retry in {wait_s}s ({i + 1}/{attempts - 1}).")
            time.sleep(wait_s)


all_stems = sorted(p.stem for p in _iterdir_retry(drive_train_im) if p.is_file())
print(f"Found {len(all_stems)} pairs total on Drive.")

# PERSISTENT split: on the first run the val list is written to Drive; all later
# runs use the SAME val set (even if the dataset grows — new stems go to TRAIN).
val_stems_json = Path(DRIVE_ROOT) / DRIVE_STATUS_SUBDIR / "val_stems.json"
train_stems, val_stems = tcl.load_or_create_val_split(
    all_stems, seed=SEED, val_fraction=VAL_FRACTION, persist_path=val_stems_json
)
print(f"Split (seed={SEED}, persistent list: {val_stems_json}): "
      f"TRAIN={len(train_stems)}, VAL_HOLDOUT={len(val_stems)} (first-run target share: {VAL_FRACTION * 100:.1f}%)")

# CAUTION: we DELIBERATELY put VAL_HOLDOUT OUTSIDE the <data_root_dir>/<TASK>/
# tree (had we put it as a "sibling" folder inside the same tree, config.py's
# `datasets_all = '+'.join([ds for ds in os.listdir(data_root_dir/task) ...])`
# auto-discovery would mistake it for a training source and add it to
# `training_set` as "TRAIN+VAL_HOLDOUT" — a VAL leak! Hence VAL_HOLDOUT lives on
# a completely separate branch.
local_task_root = Path(LOCAL_DATA_ROOT) / "datasets" / "dis" / BIREFNET_TASK
local_train_im = local_task_root / "TRAIN" / "im"
local_train_gt = local_task_root / "TRAIN" / "gt"
local_val_root = Path(LOCAL_DATA_ROOT) / "val_holdout"
local_val_im = local_val_root / "im"
local_val_gt = local_val_root / "gt"
for d in (local_train_im, local_train_gt, local_val_im, local_val_gt):
    d.mkdir(parents=True, exist_ok=True)

t0 = time.time()

def _n_files(d):
    return sum(1 for p in d.iterdir() if p.is_file()) if d.is_dir() else 0

# --- FAST PATH: tar shards (output of training/veri_tar_paketleme_hucresi.py) ---
# Copying 52k+ small files one by one over Drive FUSE takes ~75 min (with the
# occasional transient 'Errno 5'); copying a few LARGE tars and extracting them
# locally takes ~10 min. If there is NO manifest, the else branch below keeps the
# old copy_pairs path unchanged.
tar_manifest_path = drive_data_dir / "tar" / "_manifest.json"
if tar_manifest_path.exists():
    manifest = json.loads(tar_manifest_path.read_text())
    total_tar_pairs = tcl.validate_tar_manifest(manifest)  # internal consistency: sum of shard 'pairs' == total_pairs
    if total_tar_pairs > len(all_stems):
        print(f"WARNING: the manifest expects {total_tar_pairs} pairs but Drive TRAIN has {len(all_stems)} — "
              f"the dataset may have SHRUNK after packing; consider re-running the packing cell.")
    # Idempotency (session restart safety): if the local disk (TRAIN + val_holdout
    # combined) already has the number of files the manifest expects, download/extract is SKIPPED.
    n_local = _n_files(local_train_im) + _n_files(local_val_im)
    if n_local >= total_tar_pairs:
        print(f"Tar download/extract SKIPPED: {n_local} im files already local (>= manifest {total_tar_pairs}).")
    else:
        tar_cache = Path("/content/tar_cache")
        tar_cache.mkdir(parents=True, exist_ok=True)
        for sh in manifest["shards"]:
            src, dst = drive_data_dir / "tar" / sh["name"], tar_cache / sh["name"]
            if not (dst.exists() and dst.stat().st_size == sh["bytes"]):
                shutil.copy2(src, dst)  # one BIG file — fast on Drive FUSE
                if dst.stat().st_size != sh["bytes"]:
                    raise RuntimeError(
                        f"{sh['name']}: the size copied from Drive ({dst.stat().st_size}) does not match the "
                        f"manifest's ({sh['bytes']}) — the transfer may have been cut short, re-run the cell."
                    )
            with tarfile.open(dst) as tf:
                tf.extractall(local_task_root / "TRAIN", filter="data")  # members: im/<file> + gt/<file>
            dst.unlink()  # the extracted shard's local tar is deleted immediately (disk safety)
            print(f"{sh['name']}: copied + extracted ({sh['pairs']} pairs, {sh['bytes'] / 1e9:.2f} GB).")
        n_im, n_gt = _n_files(local_train_im), _n_files(local_train_gt)
        if n_im != n_gt or n_im < total_tar_pairs:
            raise RuntimeError(
                f"tar extraction does not match the manifest: im={n_im}, gt={n_gt}, expected at least {total_tar_pairs} "
                f"(and im == gt) — shards may be missing/corrupt; re-run the packing cell."
            )
    # VAL split (the existing persistent-split behavior is PRESERVED): the tars
    # extract ALL pairs into TRAIN; the stems in val_stems.json are MOVED from
    # TRAIN to val_holdout/{im,gt} (they do NOT STAY in TRAIN — no leak). This
    # block also runs in the skip case: a half-finished previous session may have
    # extracted the tars but failed to move.
    train_im_by_stem = {p.stem: p for p in local_train_im.iterdir() if p.is_file()}
    train_gt_by_stem = {p.stem: p for p in local_train_gt.iterdir() if p.is_file()}
    n_moved = 0
    for stem in val_stems:
        pi, pg = train_im_by_stem.get(stem), train_gt_by_stem.get(stem)
        if pi is not None:
            pi.replace(local_val_im / pi.name)
        if pg is not None:
            pg.replace(local_val_gt / pg.name)
        if pi is not None or pg is not None:
            n_moved += 1
    print(f"VAL_HOLDOUT: {n_moved} pairs moved from TRAIN to val_holdout (persistent list: {len(val_stems)} stems).")
    # New pairs added to Drive AFTER the tar packing — the rule is PRESERVED: new
    # stems always go to TRAIN (the val set stays fixed via the persistent list).
    # The missing ones are computed from the local listing; only the DELTA is
    # fetched with size-validated copy_pairs.
    have_train = {p.stem for p in local_train_im.iterdir() if p.is_file()}
    have_val = {p.stem for p in local_val_im.iterdir() if p.is_file()}
    delta_train = [s for s in train_stems if s not in have_train]
    delta_val = [s for s in val_stems if s not in have_val]
    n_train_copied = tcl.copy_pairs(delta_train, drive_train_im, drive_train_gt, local_train_im, local_train_gt)
    n_val_copied = tcl.copy_pairs(delta_val, drive_train_im, drive_train_gt, local_val_im, local_val_gt)
    print(f"Tar path DONE: TRAIN delta +{n_train_copied}/{len(delta_train)}, "
          f"VAL_HOLDOUT delta +{n_val_copied}/{len(delta_val)} — total {time.time() - t0:.0f} s")
else:
    # --- SLOW PATH (backward compatibility): the old file-by-file copy, UNCHANGED ---
    print(f"NOTE: no tar manifest ({tar_manifest_path}) — falling back to the file-by-file copy_pairs path. "
          f"If training/veri_tar_paketleme_hucresi.py is run ONCE on a free CPU Colab, "
          f"this step gets ~7x faster (~75 min -> ~10 min).")
    # The copy logic lives in train_colab_lib.copy_pairs (unit-tested): both the im
    # and the gt sizes are validated — truncated gt files get repaired too.
    n_train_copied = tcl.copy_pairs(train_stems, drive_train_im, drive_train_gt, local_train_im, local_train_gt)
    n_val_copied = tcl.copy_pairs(val_stems, drive_train_im, drive_train_gt, local_val_im, local_val_gt)
    print(f"Copied: TRAIN +{n_train_copied} (total {len(train_stems)}), "
          f"VAL_HOLDOUT +{n_val_copied} (total {len(val_stems)}) — {time.time() - t0:.0f} s")

EVAL_STEMS = tcl.fixed_eval_subset(val_stems, seed=SEED, n=QUICK_EVAL_N)
print(f"Selected a FIXED set of {len(EVAL_STEMS)} images for quick evaluation (the SAME images every N_EVAL_EVERY epochs).")

## (d) Model + Checkpoint/Resume Setup

If `RESUME="auto"`, it first looks for the latest `epoch_<N>.pth` on **Drive**
(`bg-remover-checkpoints/`), and failing that in the **local** checkpoint
directory (`train_colab_lib.find_latest_checkpoint` — extracts the epoch
number from the file name with a regex and picks the largest). If found,
training continues from it (model + optimizer + lr_scheduler + epoch counter);
if not, it starts from SCRATCH (from the HR-matting weights) with
`BiRefNet.from_pretrained(HF_MODEL_ID)` — the README's own recommended
"code from GitHub, weights from HuggingFace" method.

In [ ]:
import torch

sys.path.insert(0, BIREFNET_DIR)
os.chdir(BIREFNET_DIR)  # for config.py's own train.sh discovery + relative imports

from config import Config  # noqa: E402
from models.birefnet import BiRefNet  # noqa: E402
from utils import check_state_dict, set_seed  # noqa: E402

config = Config()
assert config.task == BIREFNET_TASK
# `config.training_set` is auto-discovered inside `Config.__init__` by joining
# ALL subfolders under `data_root_dir/<TASK>/` (except testsets) with '+'
# (see the config.py `datasets_all` line). In cell (c) we put VAL_HOLDOUT
# OUTSIDE this tree (see that cell's comment), but to BE SURE we still pin
# `training_set` EXPLICITLY to "TRAIN" here — rather than trusting auto-discovery.
if config.training_set != "TRAIN":
    print(f"WARNING: auto-discovery found training_set='{config.training_set}' — forcing to 'TRAIN'.")
config.training_set = "TRAIN"
print(f"config: task={config.task} size={config.size} batch_size={config.batch_size} "
      f"lr(official formula, no accum yet)={config.lr:.2e} finetune_last_epochs={config.finetune_last_epochs}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("WARNING: CUDA not found — in Colab check Runtime > Change runtime type > Hardware accelerator > A100.")
set_seed(config.rand_seed if config.rand_seed else SEED)

local_ckpt_dir = Path("/content/ckpts")
local_ckpt_dir.mkdir(parents=True, exist_ok=True)
drive_ckpt_dir = Path(DRIVE_ROOT) / DRIVE_CKPT_SUBDIR
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)

resume_info = None
if RESUME == "auto":
    resume_info = tcl.find_latest_checkpoint(drive_ckpt_dir) or tcl.find_latest_checkpoint(local_ckpt_dir)
elif RESUME:
    m = re.search(r"epoch_(\d+)", str(RESUME))
    assert m, f"no 'epoch_<N>' pattern found in the RESUME path: {RESUME}"
    resume_info = (RESUME, int(m.group(1)))

payload = None
if resume_info:
    ckpt_path_str, resumed_epoch = resume_info
    local_resume_path = local_ckpt_dir / Path(ckpt_path_str).name
    if str(local_resume_path) != ckpt_path_str:
        shutil.copy2(ckpt_path_str, local_resume_path)  # Drive -> local disk (one file, fast) — so the next torch.load reads locally
    print(f"Checkpoint found: {ckpt_path_str} (epoch {resumed_epoch}) -> resuming.")
    payload = torch.load(local_resume_path, map_location="cpu", weights_only=False)
    assert isinstance(payload, dict) and "model" in payload, (
        f"Checkpoint is not in the expected bundle: {ckpt_path_str}\n"
        f"The epoch_N.pth files this notebook produces are dicts with the "
        f"{{'model','optimizer','lr_scheduler','epoch'}} keys. Your file is probably a RAW state_dict "
        f"(e.g. epoch_244.pth downloaded from the official BiRefNet releases) — these cannot be RESUMEd "
        f"directly because they carry no optimizer/lr_scheduler/epoch info. If you want to start from raw "
        f"weights, remove the file from the checkpoint directories and use HF_MODEL_ID "
        f"(RESUME='auto' + an empty directory = starts from scratch)."
    )
    model = BiRefNet(bb_pretrained=False)
    model.load_state_dict(check_state_dict(payload["model"]))
    epoch_st = payload.get("epoch", resumed_epoch) + 1
else:
    print(f"No checkpoint — starting from SCRATCH from the {HF_MODEL_ID} initial weights.")
    model = BiRefNet.from_pretrained(HF_MODEL_ID)
    epoch_st = 1

model = model.to(device)
if config.precisionHigh:
    torch.set_float32_matmul_precision("high")

# torch.compile is LAZY: the wrapper call here almost never fails; the real
# compilation is triggered at the FIRST BATCH — so a try/except here alone would
# be a misleading guarantee. Therefore: (1) the default is COMPILE_MODEL=False
# (a safe baseline for the first run), (2) if True is selected, dynamo's
# suppress_errors is enabled — graph pieces that fail to compile silently fall
# back to eager mode instead of CRASHING the training (the speedup is lost for
# those pieces, correctness is unaffected).
if COMPILE_MODEL:
    import torch._dynamo

    torch._dynamo.config.suppress_errors = True
    model = torch.compile(model, mode="default")
    print("torch.compile enabled (suppress_errors=True — pieces that fail to compile fall back to eager).")
else:
    print("torch.compile OFF (COMPILE_MODEL=False — the safe default for a first run).")

print(f"Starting epoch: {epoch_st} / target {EPOCHS}")

## (e) Category-Weighted Sampling (`WeightedRandomSampler`)

The physical multipliers of `scripts/make_composites.py` (transparent x10,
camouflage x2 — see that file's docstring and
the project's internal phase report (removed from the repo) §2) do not by themselves guarantee the
>=20% target (the real Colab counts only become known AFTER materialization).
The **least invasive** solution: without changing the official
`MyData`/`train.py` code at all, only replace the `DataLoader`'s `sampler=`
(instead of `shuffle=is_train, sampler=None`) with a `WeightedRandomSampler` —
the weights are computed from Phase 2's composite manifest
(`train_composites_manifest.jsonl`, id->category, copied to Drive by
`stage7_drive_copy`) (see `train_colab_lib.compute_sample_weights` — targeted
categories get exactly their `target_share`; the others keep their raw
proportions among themselves).

We DELIBERATELY rejected physical oversampling (raising the multiplier to
13-15x, the first draft of the Phase 2 report): the disk/IO cost is high AND
the final size of the `general` category (0-4000, see phase-2 report §4) was
unknown in advance — the sampler weights adjust automatically to the REAL
counts, no physical data regeneration needed.

**v2:** The target shares are now selected via the `SAMPLER_PRESET` parameter
(`"v1"`/`"v2"`) — `tcl.SAMPLER_PRESETS[SAMPLER_PRESET]` (see
`train_colab_lib.SAMPLER_PRESET_V1`/`SAMPLER_PRESET_V2`). v1 targeting only
transparent+camouflage at >=20% and leaving complex/thin/hair untargeted
(most of the remaining share going to hair, whose raw volume is much larger)
was the root cause of the catastrophic forgetting in the v1 comparison
report; v2 fixes this by giving hair/complex/thin EXPLICIT targets too.


In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler

from dataset import MyData  # noqa: E402  (BIREFNET_DIR is on sys.path, from cell (d))

train_dataset = MyData(datasets=config.training_set, data_size=config.size, is_train=True)
print(f"TRAIN dataset: {len(train_dataset)} samples.")

manifest_path = drive_data_dir / "train_composites_manifest.jsonl"
assert manifest_path.exists(), f"Composite manifest not found: {manifest_path}"
stem_category = tcl.load_stem_categories(manifest_path)

stems_in_order = [Path(p).stem for p in train_dataset.image_paths]

# --- v3 pre-flight guard (reviewer finding #2): the ENTIRE point of the v3
# preset is that the _o00 (original background) copies produced by
# v3_veri_guncelleme_hucresi.py are seen in training. If the Drive data has not
# been updated yet, this EXPENSIVE GPU run would silently miss the whole point
# of v3 — we stop early and loudly. ---
o00_count = sum(1 for s in stems_in_order if s.endswith("_o00"))
print(f"_o00 (original background) sample count: {o00_count} / total {len(stems_in_order)}")
if SAMPLER_PRESET in ("v3", "v4", "v5", "v7") and o00_count == 0:
    raise RuntimeError(
        "v3 data is missing — run v3_veri_guncelleme_hucresi.py in a free CPU session first "
        f"(no _o00 files found in bg-remover-data/TRAIN on Drive; SAMPLER_PRESET={SAMPLER_PRESET!r} "
        "is meaningless without this data — stopped to avoid wasting an expensive GPU session)."
    )

# --- v4 pre-flight guard (SAME purpose as the v3 guard): if the v4 preset's new
# categories (text/fx/illustration — produced by v4_veri_guncelleme_hucresi.py)
# are missing from the manifest+TRAIN, their 26% combined target share would go
# to waste and this EXPENSIVE GPU run would silently miss the whole point of
# v4 — we stop early and loudly (at least 100 samples required from each new
# category). ---
if SAMPLER_PRESET in ("v4", "v5", "v7"):
    _v4_counts = {c: 0 for c in ("text", "fx", "illustration")}
    for _s in stems_in_order:
        _c = stem_category.get(_s)
        if _c in _v4_counts:
            _v4_counts[_c] += 1
    print(f"v4 new-category sample counts (manifest+TRAIN intersection): {_v4_counts}")
    _v4_missing = {c: n for c, n in _v4_counts.items() if n < 100}
    if _v4_missing:
        raise RuntimeError(
            "v4 data is missing — run v4_veri_guncelleme_hucresi.py in a free CPU session first "
            f"(at least 100 samples required from EACH of the text/fx/illustration categories; missing: {_v4_missing}; "
            "SAMPLER_PRESET={SAMPLER_PRESET!r} is meaningless without this data — stopped to avoid wasting an expensive GPU session)."
        )

n_unknown = sum(1 for s in stems_in_order if s not in stem_category)
if n_unknown:
    print(f"WARNING: {n_unknown}/{len(stems_in_order)} stems not found in the manifest "
          f"(category '_other' will be assumed — the sampler still works, these "
          f"samples are just excluded from the targeted-category share computation).")

assert SAMPLER_PRESET in tcl.SAMPLER_PRESETS, (
    f"unknown SAMPLER_PRESET={SAMPLER_PRESET!r} — valid: {sorted(tcl.SAMPLER_PRESETS)}"
)
CATEGORY_TARGET_SHARE = tcl.SAMPLER_PRESETS[SAMPLER_PRESET]
print(f"SAMPLER_PRESET={SAMPLER_PRESET!r} -> target shares: {CATEGORY_TARGET_SHARE}")

sample_weights = tcl.compute_sample_weights(stems_in_order, stem_category, CATEGORY_TARGET_SHARE)
achieved_shares = tcl.compute_expected_shares(sample_weights, stems_in_order, stem_category)
print("EXPECTED in-epoch category shares after building the sampler:")
for cat, share in sorted(achieved_shares.items(), key=lambda kv: -kv[1]):
    print(f"  {cat}: {share * 100:.1f}%")

In [ ]:
# Tolerance: the SAMPLER_PRESET sum is deliberately <1.0 (the remaining share is reserved
# for "_other" stems whose category is missing from the manifest — see the
# train_colab_lib.SAMPLER_PRESET_V2 docstring). If ALL targeted categories are present in
# this run AND there are no "_other" stems at all (n_unknown=0, the common case), that
# remaining share goes to no sample — since WeightedRandomSampler works only with RELATIVE
# weights, this leads to a HARMLESS rescaling of the targets among themselves by about
# (1-sum(target_share))% (v1's sum=40% never showed this effect because the untargeted
# "other" categories — hair/complex/thin/general — were always present). Hence the
# tolerance is 2e-2 instead of 1e-6 (v2 prep report: "within 1%" + a safety margin).
_TARGET_SHARE_TOLERANCE = 1e-6 if SAMPLER_PRESET == "v1" else 2e-2
for cat, target in CATEGORY_TARGET_SHARE.items():
    if cat in achieved_shares:
        assert abs(achieved_shares[cat] - target) < _TARGET_SHARE_TOLERANCE, (
            f"expected share for {cat} is {target*100:.1f}% but computed {achieved_shares[cat]*100:.1f}% "
            f"— the train_colab_lib.compute_sample_weights logic needs checking."
        )

_epoch_num_samples = tcl.resolve_sampler_num_samples(len(train_dataset), EPOCH_NUM_SAMPLES)
print(f"Samples per epoch: {_epoch_num_samples} "
      f"(EPOCH_NUM_SAMPLES={EPOCH_NUM_SAMPLES!r}, dataset size={len(train_dataset)}).")
sampler = WeightedRandomSampler(sample_weights, num_samples=_epoch_num_samples, replacement=True)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH,
    sampler=sampler,
    shuffle=False,  # CANNOT be combined with shuffle=True when a sampler is given (torch constraint) — the sampler already shuffles.
    num_workers=NUM_WORKERS,  # DELIBERATE DEVIATION: the official formula min(num_workers, batch_size) would drop
                              # to 2 workers at bs=2 — data reading becomes the bottleneck in a ~13.7k-iteration
                              # epoch; with A100 VMs having plenty of CPU we decouple the worker count from the batch.
    pin_memory=True,
    drop_last=True,  # SAME as the official train.py (the last incomplete batch is dropped).
)
print(f"train_loader ready: {len(train_loader)} batches/epoch (batch_size={BATCH}, accum={ACCUM} -> effective batch={BATCH * ACCUM}).")

## (f) Loss Function, Optimizer, Quick Evaluation

`PixLoss`/`ClsLoss` are used UNCHANGED from the official `loss.py`.
`effective_lr` (when LR=None) adapts the official formula (`config.py`) to the
effective batch (`BATCH*ACCUM`) (see the `train_colab_lib.effective_lr`
docstring — gradient accumulation does not exist in the official code, so this
adaptation is OUR addition).

`run_quick_eval` uses the inference pattern `bgr/segmenter.py` ALREADY uses
(`model(inp)[-1].sigmoid()`, the structure `BiRefNet.forward` returns when
`self.training=False`) and the project's shared metrics module
(`benchmark.metrics.mae`) — an approximate MAE over the fixed `EVAL_STEMS`
from VAL_HOLDOUT, at training resolution (1024x1024). **This does NOT REPLACE
the full 155-image benchmark** — it is only a quick trend signal between
epochs; the full measurement happens locally after training
(`benchmark/run.py`).

In [ ]:
from loss import ClsLoss, PixLoss  # noqa: E402
from torchvision import transforms

pix_loss = PixLoss()
cls_loss = ClsLoss()
criterion_gdt = torch.nn.BCELoss() if config.out_ref else None
BASE_LAMBDAS_PIX_LAST = dict(pix_loss.lambdas_pix_last)  # so the finetune-reweight is recomputed EVERY epoch from THIS baseline (see the training loop cell's note)

lr = tcl.effective_lr(config.task, BATCH, ACCUM, base_lr_override=LR)
if LR is None:  # scaled only when the official FORMULA is used — a hand-entered LR is used AS IS.
    lr *= LR_SCALE
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-2)
lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[m if m > 0 else EPOCHS + m + 1 for m in config.lr_decay_epochs],
    gamma=config.lr_decay_rate,
)
if payload is not None and "optimizer" in payload:
    optimizer.load_state_dict(payload["optimizer"])
    lr_scheduler.load_state_dict(payload["lr_scheduler"])
    print("Optimizer + lr_scheduler state restored from the checkpoint.")
print(f"Learning rate: {lr:.3e} (LR parameter={'auto' if LR is None else LR}, "
      f"LR_SCALE={LR_SCALE if LR is None else 'not applied (LR set by hand)'})")

_quick_eval_transform = transforms.Compose([
    transforms.Resize(config.size[::-1]),  # config.size = (wid, hei); Resize expects (hei, wid).
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


@torch.no_grad()
def run_quick_eval(model, eval_stems, im_dir, gt_dir, device) -> float:
    import numpy as np
    from PIL import Image

    model.eval()
    maes = []
    for stem in eval_stems:
        img_path, gt_path = im_dir / f"{stem}.jpg", gt_dir / f"{stem}.png"
        if not (img_path.exists() and gt_path.exists()):
            continue
        img = Image.open(img_path).convert("RGB")
        gt = Image.open(gt_path).convert("L").resize(config.size, Image.BILINEAR)
        inp = _quick_eval_transform(img).unsqueeze(0).to(device)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            pred = model(inp)[-1].sigmoid()
        pred_np = pred.float().cpu().squeeze().numpy()
        gt_np = np.asarray(gt, dtype=np.float32) / 255.0
        maes.append(bm.mae(pred_np, gt_np))
    model.train()
    return float(sum(maes) / len(maes)) if maes else float("nan")


print(f"Ready: quick-evaluation function defined over {len(EVAL_STEMS)} fixed VAL_HOLDOUT images.")

## (g) Training Loop — Checkpoint/Resume + Log + Periodic Quick Evaluation

At the end of every epoch:
1. A row is appended to `MyDrive/bg-remover-status/train_log.txt` (`epoch,
   loss, lr, time` — the controller watches this file FROM OUTSIDE) + the
   measured epoch duration and approximate Colab unit cost are printed to the
   console (from the first epoch on — compare the theoretical estimate in the
   cost table against the real measurement).
2. The checkpoint (model+optimizer+lr_scheduler+epoch) is written FIRST to the
   local disk, THEN to Drive (`bg-remover-checkpoints/epoch_<N>.pth`); in both
   places only the last `KEEP_LAST_N_CHECKPOINTS` are kept
   (`train_colab_lib.prune_old_checkpoints`).
3. Every `N_EVAL_EVERY` epochs (+ ALWAYS on the last epoch) the
   quick-evaluation MAE is computed and appended to the log.

**Drive resilience:** writing to the Drive mount (the checkpoint copy, the log
row) can occasionally throw a transient I/O error (quota, sync stall). These
do NOT KILL the training: the local checkpoint is already saved, and the Drive
copy/log row is wrapped in try/except — a warning is printed and it is retried
on the NEXT epoch (since the checkpoint is copied every epoch anyway, the loss
is at most one epoch of Drive lag). Writing the FATAL traceback is
best-effort too.

**Drive quota:** each checkpoint bundle (model fp32 + AdamW moments x2 +
scheduler) is ~2.5-3GB; with `KEEP_LAST_N_CHECKPOINTS=3` it occupies **~8-9GB**
on Drive. Make sure your Drive has that much free space (that is most of the
free 15GB quota — lower the parameter to 2 if needed).

**torch.compile note:** if `COMPILE_MODEL=True`, the `model.eval()` <->
`model.train()` transition (the quick-evaluation cell) triggers one more
compilation THE FIRST TIME (the eval graph differs from train) — it is normal
for the first quick evaluation to take a few minutes longer; later ones run
from the cache.

**The fine-tune trick (official `train.py::train_epoch`, Matting branch) —
made resume-safe:** in the last `|finetune_last_epochs|` (Matting: 10) epochs
the official code multiplies the `ssim`/`mse` weights CUMULATIVELY by x0.9
every epoch — but that logic rests on the assumption of a single run where
`PixLoss()` stays alive THE WHOLE TIME and never restarts. In our
multi-session scenario (a Colab drop every few hours) `PixLoss()` is rebuilt
FROM SCRATCH every session — imitating the cumulative multiplication as-is
would be WRONG (the counter resets on every resume and the exponential decay
breaks). Instead we recompute EVERY epoch from the baseline
(`BASE_LAMBDAS_PIX_LAST`) with `0.9 ** n` based on the ABSOLUTE epoch number —
MATHEMATICALLY IDENTICAL to the official code running uninterrupted, but
robust to resumes.

In [ ]:
import time
import traceback

STATUS_DIR = Path(DRIVE_ROOT) / DRIVE_STATUS_SUBDIR
TRAIN_LOG_PATH = STATUS_DIR / "train_log.txt"
STATUS_DIR.mkdir(parents=True, exist_ok=True)

UNITS_PER_HOUR_A100 = 13  # approximate (Colab A100 ~11-13 units/hour); verify the exact value in Colab's "Resources" panel.


def log_epoch_row(epoch: int, loss: float, lr_now: float, elapsed_sec: float, eval_mae: float | None) -> None:
    row = f"epoch={epoch}\tloss={loss:.6f}\tlr={lr_now:.8f}\ttime_sec={elapsed_sec:.1f}"
    if eval_mae is not None:
        row += f"\teval_mae={eval_mae:.6f}"
    print(row)
    # Writing the log to Drive is best-effort: a transient Drive I/O error must NOT
    # KILL the training (the row was already printed to the console; the next
    # epoch's row will be attempted again).
    try:
        with open(TRAIN_LOG_PATH, "a") as f:
            f.write(row + "\n")
    except OSError as e:
        print(f"  WARNING: could not write to train_log.txt ({e}) — training continues, will retry next epoch.")


def save_and_sync_checkpoint(epoch: int) -> None:
    raw_state = model.state_dict()  # with torch.compile the keys may carry the '_orig_mod.' prefix -- SAME behavior as
                                     # the official train.py (saves WITHOUT stripping the prefix); check_state_dict is
                                     # always applied at load time.
    payload_out = {
        "model": raw_state,
        "optimizer": optimizer.state_dict(),
        "lr_scheduler": lr_scheduler.state_dict(),
        "epoch": epoch,
    }
    # 1) Local disk FIRST — this must always succeed (if it fails there is a real problem, the error propagates).
    local_path = local_ckpt_dir / f"epoch_{epoch}.pth"
    torch.save(payload_out, local_path)
    tcl.prune_old_checkpoints(local_ckpt_dir, KEEP_LAST_N_CHECKPOINTS)

    # 2) THEN Drive — best-effort: a transient Drive I/O error (quota/sync stall)
    # does not kill the training; the local copy is safe and the NEXT epoch will
    # attempt a new Drive copy (worst case, Drive lags one epoch behind).
    try:
        drive_path = drive_ckpt_dir / f"epoch_{epoch}.pth"
        shutil.copy2(local_path, drive_path)
        tcl.prune_old_checkpoints(drive_ckpt_dir, KEEP_LAST_N_CHECKPOINTS)
        print(f"  checkpoint saved + copied to Drive: {drive_path}")
    except OSError as e:
        print(f"  WARNING: could not copy checkpoint to Drive ({e}) — the LOCAL copy is safe: {local_path}; "
              f"will retry next epoch.")


def train_one_epoch(epoch: int) -> float:
    model.train()

    # --- Fine-tune trick: recompute FROM THE BASELINE by absolute epoch number (resume-safe, see the note above) ---
    pix_loss.lambdas_pix_last = dict(BASE_LAMBDAS_PIX_LAST)
    if tcl.should_apply_finetune_reweight(epoch, EPOCHS, config.finetune_last_epochs):
        n = epoch - (EPOCHS + config.finetune_last_epochs)
        if config.task == "Matting":
            pix_loss.lambdas_pix_last["mse"] = BASE_LAMBDAS_PIX_LAST["mse"] * (0.9 ** n)
            pix_loss.lambdas_pix_last["ssim"] = BASE_LAMBDAS_PIX_LAST["ssim"] * (0.9 ** n)
        else:
            pix_loss.lambdas_pix_last["bce"] = BASE_LAMBDAS_PIX_LAST["bce"] * 0
            pix_loss.lambdas_pix_last["iou"] = BASE_LAMBDAS_PIX_LAST["iou"] * (0.5 ** n)
            pix_loss.lambdas_pix_last["mae"] = BASE_LAMBDAS_PIX_LAST["mae"] * (0.9 ** n)

    running_sum, running_n = 0.0, 0
    n_batches = len(train_loader)
    optimizer.zero_grad()
    for micro_step, batch in enumerate(train_loader):
        inputs = batch[0].to(device, non_blocking=True)
        gts = batch[1].to(device, non_blocking=True)
        class_labels = batch[2].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            scaled_preds, class_preds_lst = model(inputs)
            if config.out_ref:
                (outs_gdt_pred, outs_gdt_label), scaled_preds = scaled_preds

        # why: BCE (BCELoss / binary_cross_entropy) is forbidden under autocast; SAME math as the official
        # BiRefNet train.py/loss.py, only this block (gdt + pix_loss) is computed in fp32.
        with torch.autocast(device_type="cuda", enabled=False):
            if config.out_ref:
                loss_gdt = None
                for gi, (gp, gl) in enumerate(zip(outs_gdt_pred, outs_gdt_label)):
                    gp_i = torch.nn.functional.interpolate(gp.float(), size=gl.shape[2:], mode="bilinear", align_corners=True).sigmoid()
                    gl_i = gl.float().sigmoid()
                    li = criterion_gdt(gp_i, gl_i)
                    loss_gdt = li if gi == 0 else loss_gdt + li
            loss_cls = 0.0 if None in class_preds_lst else cls_loss(class_preds_lst, class_labels)
            # pix_loss (the official PixLoss) also uses raw BCELoss in its 'bce' component -- promoted to fp32 for the same reason.
            scaled_preds_f = [sp.float() for sp in scaled_preds]
            loss_pix, _loss_dict_pix = pix_loss(scaled_preds_f, torch.clamp(gts, 0, 1).float(), pix_loss_lambda=1.0)
            loss = loss_pix + loss_cls
            if config.out_ref:
                loss = loss + loss_gdt * 1.0

        (loss / ACCUM).backward()
        if (micro_step + 1) % ACCUM == 0 or (micro_step + 1) == n_batches:
            optimizer.step()
            optimizer.zero_grad()

        running_sum += loss.item() * inputs.size(0)
        running_n += inputs.size(0)
        if micro_step % 200 == 0:
            print(f"  epoch {epoch} iter {micro_step}/{n_batches} loss={loss.item():.5g}")

    lr_scheduler.step()
    return running_sum / max(running_n, 1)


def main() -> None:
    for epoch in range(epoch_st, EPOCHS + 1):
        t0 = time.time()
        avg_loss = train_one_epoch(epoch)
        elapsed = time.time() - t0
        current_lr = optimizer.param_groups[0]["lr"]

        eval_mae = None
        if epoch % N_EVAL_EVERY == 0 or epoch == EPOCHS:
            eval_mae = run_quick_eval(model, EVAL_STEMS, local_val_im, local_val_gt, device)

        log_epoch_row(epoch, avg_loss, current_lr, elapsed, eval_mae)
        save_and_sync_checkpoint(epoch)

        # MEASURED cost report (compare with the theoretical table in the parameters cell):
        hours = elapsed / 3600
        est_units = hours * UNITS_PER_HOUR_A100
        remaining = EPOCHS - epoch
        print(f"  COST: this epoch {hours:.2f} hours ~ {est_units:.0f} units "
              f"(assuming A100 ~{UNITS_PER_HOUR_A100} units/hour); "
              f"remaining {remaining} epochs ~ {remaining * hours:.1f} hours ~ {remaining * est_units:.0f} units. "
              f"If this will exceed your budget, stop now — RESUME='auto' continues from where it left off.")

    print("TRAINING COMPLETE.")


try:
    main()
except Exception:
    tb = traceback.format_exc()
    print(tb)
    try:  # the FATAL record is best-effort too — if Drive is unwritable it must not shadow the real error.
        with open(TRAIN_LOG_PATH, "a") as f:
            f.write(f"epoch=FATAL\ttraceback={tb!r}\n")
    except OSError as log_err:
        print(f"WARNING: could not write the FATAL record to train_log.txt ({log_err}).")
    raise

## After Training

- **The full benchmark (155 images) runs LOCALLY (M4)** — this notebook's job
  is only the fine-tune + the quick MAE trend. When training finishes,
  download the latest checkpoint from Drive
  (`bg-remover-checkpoints/epoch_<EPOCHS>.pth`) and evaluate locally through
  the `scripts/export_birefnet.py`/`benchmark/run.py` flow (see the
  the project's internal phase report (removed from the repo) §5 targets: camouflage MAE <=0.075,
  transparent MAE <0.06).
- The controller already watches `train_log.txt`
  (`MyDrive/bg-remover-status/`) from outside; re-running this notebook
  (`RESUME="auto"`) continues from where it left off.
- Phase 4 (packaging/ONNX export) will use this checkpoint as input.